# GPX Command-Line Tutorial

This notebook walks through a complete GPX workflow:
1. Create synthetic tabular training data
2. Train a surrogate model with `gpx fit`
3. Inspect model I/O with `gpx spec`
4. Assess model quality with `gpx qa`
5. Predict on new points with `gpx predict`
6. Generate and use a Python helper with `gpx py`

## Preamble: Install `gpx` CLI

Before running the tutorial, install the `gpx` executable from the official release distribution:
https://github.com/relf/EGObox/releases

## Setup

We keep tutorial artifacts in `notebooks/gpx_cli_tutorial` and run GPX **directly** with shell commands (`!gpx ...`) in notebook cells.

In [1]:
import numpy as np
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "Cargo.toml").exists() and (ROOT.parent / "Cargo.toml").exists():
    ROOT = ROOT.parent

WORKDIR = ROOT / "doc" / "gpx_cli_tutorial_out"
WORKDIR.mkdir(parents=True, exist_ok=True)
print("WORKDIR:", WORKDIR)

# Use the artifact folder as current working directory so shell commands are concise.
%cd $WORKDIR

WORKDIR: d:\rlafage\workspace\egobox\doc\gpx_cli_tutorial_out
d:\rlafage\workspace\egobox\doc\gpx_cli_tutorial_out


## 1) Create Synthetic Training Data

We build a simple **2-input / 2-output** dataset:

\[
y_1 = (1-x_1)^2 + 100(x_2 - x_1^2)^2
\]

\[
y_2 = \sin(x_1) + 0.5x_2^2
\]

Training table layout is `x1,x2,y1,y2`.

In [2]:
rng = np.random.default_rng(42)
n_train = 40

x1 = rng.uniform(-2.0, 2.0, n_train)
x2 = rng.uniform(-1.0, 3.0, n_train)
y1 = (1.0 - x1) ** 2 + 100.0 * (x2 - x1 ** 2) ** 2
y2 = np.sin(x1) + 0.5 * x2 ** 2

train = np.column_stack([x1, x2, y1, y2])
train_file = WORKDIR / "training_data.csv"
np.savetxt(train_file, train, delimiter=",", header="x1,x2,y1,y2", comments="")
print("Saved:", train_file, "shape:", train.shape)

Saved: d:\rlafage\workspace\egobox\doc\gpx_cli_tutorial_out\training_data.csv shape: (40, 4)


## 2) Fit Model, Inspect Spec, and Run QA

In [3]:
!gpx fit training_data.csv --outputs 2

Trained default GP surrogate model(s) from training_data.csv
Training samples: 40
Input dimension: 2
Output dimension: 2
Generated surrogate models: 2
Output columns used: last 2
Number of clusters: 1
Recombination: Smooth(None)
Smooth factor: None
Regression spec: Constant
Correlation spec: SquaredExponential
KPLS dim: None
Input format: Csv
Output format: Binary
Saved model to surrogate_model.gpx


In [4]:
!gpx spec

Model file: surrogate_model.gpx
Model count in file: 2
Spec mode: all surrogates
Surrogate models:
  - model[0]: Mixture[Smooth](Constant_SquaredExponentialGP(mean=ConstantMean, corr=SquaredExponential, theta=[0.2560509533510202, 0.022468923552421017], variance=27141645780.1258, likelihood=175.42176440332213))
    input dimension: 2
    output dimension: 1
  - model[1]: Mixture[Smooth](Constant_SquaredExponentialGP(mean=ConstantMean, corr=SquaredExponential, theta=[0.27806631916093977, 0.10880586956900448], variance=589.537334249493, likelihood=229.7315375522391))
    input dimension: 2
    output dimension: 1
Input specification:
  - supported formats: csv, npy
  - csv row layout: x1,x2,...,x2
  - npy array shape: (n_samples, 2)
  - expected input dimension: 2
Output specification:
  - output dimension (number of surrogates): 2
  - predict csv columns: y_pred1..y_pred2, y_var1..y_var2 (with --with-variance)
  - predict npy shape: (n_samples, 2)
  - predict --with-variance npy shape: (

In [10]:
!gpx qa

Loaded GP model: Mixture[Smooth](Constant_SquaredExponentialGP(mean=ConstantMean, corr=SquaredExponential, theta=[0.2560509533510202, 0.022468923552421017], variance=27141645780.1258, likelihood=175.42176440332213))
Loaded GP model: Mixture[Smooth](Constant_SquaredExponentialGP(mean=ConstantMean, corr=SquaredExponential, theta=[0.27806631916093977, 0.10880586956900448], variance=589.537334249493, likelihood=229.7315375522391))
Training data (reference model 0): 40 samples (2-dim)

IAEalpha plot data for selected GP model index 0:
Alpha | Empirical coverage | Target coverage | Delta
---------------------------------------------------
 2.00% |       92.50%      |     98.00%    |  5.50%
 7.05% |       90.00%      |     92.95%    |  2.95%
12.11% |       82.50%      |     87.89%    |  5.39%
17.16% |       80.00%      |     82.84%    |  2.84%
22.21% |       77.50%      |     77.79%    |  0.29%
27.26% |       72.50%      |     72.74%    |  0.24%
32.32% |       67.50%      |     67.68%    |  0

Check `gpx <command> --help` for further options

## 3) Predict on New Points (with Variance)

For this multi-output model, `gpx predict --with-variance` writes:
- input columns (`x1,x2`)
- prediction columns (`y_pred1,y_pred2`)
- variance columns (`y_var1,y_var2`).

In [6]:
n_pred = 12
xp1 = rng.uniform(-2.0, 2.0, n_pred)
xp2 = rng.uniform(-1.0, 3.0, n_pred)
xpred = np.column_stack([xp1, xp2])

pred_in = WORKDIR / "predict_input.csv"
pred_out = WORKDIR / "predictions.csv"
np.savetxt(pred_in, xpred, delimiter=",", header="x1,x2", comments="")

!gpx predict predict_input.csv --output predictions.csv --with-variance

table = np.loadtxt(pred_out, delimiter=",", skiprows=1)
print("prediction table shape:", table.shape)

x_cols = table[:, :2]
y_pred = table[:, 2:4]
y_var = table[:, 4:6]

print("inputs shape:", x_cols.shape)
print("predictions shape:", y_pred.shape)
print("variances shape:", y_var.shape)
print("first 3 predicted rows:\n", y_pred[:3])
print("first 3 variance rows:\n", y_var[:3])

Predicted 12 sample(s) from predict_input.csv and wrote predictions.csv
Model file: surrogate_model.gpx (all 2 models)
Input format: Csv
Output format: Csv
Output columns: x1..x2, y_pred1..y_pred2, y_var1..y_var2
prediction table shape: (11, 6)
inputs shape: (11, 2)
predictions shape: (11, 2)
variances shape: (11, 2)
first 3 predicted rows:
 [[ 1.66143251e+01 -2.28429506e-01]
 [ 1.88245091e+02  9.72109294e-01]
 [ 1.18422537e+01  1.85267644e-01]]
first 3 variance rows:
 [[4.04082320e-04 3.10616356e-11]
 [4.19108989e-04 2.27166011e-11]
 [2.74454089e-04 1.03614042e-11]]


## 4) Generate Python Helper (`gpx py`) and Use It

The generated script embeds the model and provides `predict(x, with_variance=False, model_index=None)`.

In [7]:
py_file = WORKDIR / "gpx.py"
!gpx py -o gpx.py

try:
    from gpx import predict

    x_small = np.array([[0.0, 0.0], [1.0, 1.0]], dtype=float)
    pred, var = predict(x_small, with_variance=True)
    print("x_small:", x_small.shape)
    print("pred (multi-output):", pred.shape)
    print("var (multi-output):", var.shape)
except Exception as exc:
    print("Optional helper execution failed:", exc)
    print("This can happen if `gpx` is not available in PATH for the generated helper runtime.")

Generated Python helper script: gpx.py
Embedded model source: surrogate_model.gpx
Embedded bytes: 34077
Input dimension: 2
Output dimension (all models): 2
x_small: (2, 2)
pred (multi-output): (2, 2)
var (multi-output): (2, 2)


## Summary

You ran an end-to-end **multi-output** GPX workflow:
- `gpx fit <train.csv> --outputs 2 -o surrogate_model.gpx`
- `gpx spec --model surrogate_model.gpx`
- `gpx qa --model surrogate_model.gpx`
- `gpx predict <input.csv> --model surrogate_model.gpx --output predictions.csv --with-variance`
- `gpx py --model surrogate_model.gpx -o gpx.py`

Artifacts are stored in `notebooks/gpx_cli_tutorial_out`.